In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from dotenv import load_dotenv
from openagv.llm import create_clients

# Load environment variables
load_dotenv()

# Tool-calling model for the SK executor
_, async_client, chat_completion_service = create_clients("ollama", "qwen2.5:14b")

# Vision model for image analysis
vision_client, _, _ = create_clients("ollama", "llama3.2-vision")

In [2]:
from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.modules.vision import ORVisionAnalyzer
from openagv.modules.textcard import TextCardGenerator

instruct = UserInstruction(
    """Create a video showcasing flowers with descriptive text overlays.
    
    For each flower image:
    1. First analyze the image to get a description
    2. Generate a lower-third text overlay with the flower name and a short description
    3. Add the flower to the main timeline track (3 seconds each)
    4. Add the text overlay at the same time position on an overlay track
    
    Make sure every flower has both a background clip and a text overlay.
    Sort flowers by how vibrant/colorful they appear."""
)

# Setup AssetBin and add flower images
ab = AssetBin()
ab.add_wildcard("../assets/*.jpg")

# Initialize modules — vision uses llama3.2-vision, executor uses qwen2.5
vision_analyzer = ORVisionAnalyzer(client=vision_client, model='llama3.2-vision')
textcard = TextCardGenerator(output_dir="./text_cards", width=1920, height=1080, font_size=36)
timeline = OTIOTimeline(width=1920, height=1080, fps=30, name="Flowers with Overlays")

# Setup Executor with all modules
ex = SKLoopExecutor(
    ab, 
    instruct, 
    chat_completion=chat_completion_service, 
    uses=[vision_analyzer, textcard, timeline], 
    debug=True
)

# Execute autonomously
await ex.start()

[INFO] Starting execution...
[DEBUG] Chat History: 2 messages
[INFO] Final Agent Response: 范冰春
{"name": "AssetBin-list_assets", "arguments": {}}
</tool_call>
[INFO] Execution finished.


In [3]:
print(timeline.get_summary())
print(f"\nOverlay tracks: {timeline.get_overlay_tracks()}")

Timeline 'Flowers with Overlays' - Main track: 0 items, 0.0s

Overlay tracks: []


In [4]:
await ex.nudge(UserInstruction(
    "Please verify that every flower in the timeline has a corresponding text overlay. "
    "If any are missing, add them now."
))

[INFO] Nudged with: Please verify that every flower in the timeline has a corresponding text overlay. If any are missing, add them now.
[DEBUG] Chat History: 4 messages
[INFO] Advancing to step: AssetBin.get_all_assets_analysis
[DEBUG] Invoking AssetBin.get_all_assets_analysis with args: {}
[DEBUG] Result from AssetBin.get_all_assets_analysis: No analyses found in bin.
[INFO] Advancing to step: AssetBin.list_assets
[DEBUG] Invoking AssetBin.list_assets with args: {}
[DEBUG] Result from AssetBin.list_assets: ID: b8ef5dd5efbf3c851db53edc34f38eac7f17d3000dfe478ded645d7174498796 | File: ../assets/olia-gozha-9A_peGrSbZc-unsplash.jpg (IMAGE)
ID: 6786e06d6f727b5b8cf39ecb41c9f61ca4fe2763ff8ea066d7ce25c3f8845690 | File: ../assets/evie-s-w1JE5duY62M-unsplash.jpg (IMAGE)
ID: 25301e980a702dcb97c2376dae14f9e73b56fee9fb969a464aa4ecf277ce8a31 | File: ../assets/sergey-shmidt-koy6FlCCy5s-unsplash.jpg (IMAGE)
ID: 61415dce13017e2f4dcea0043b1ba13bc2929d151c13ad2b61937501d5718f20 | File: ../assets/zoltan-t

In [5]:
timeline.to_otio_file('flowers_overlay.otio')
print("Timeline saved to flowers_overlay.otio")

Timeline saved to flowers_overlay.otio


In [6]:
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer()
renderer.set_otio(timeline)
renderer.validate()
renderer.render('flowers_with_overlays.mp4')